<a href="https://colab.research.google.com/github/jalil7777/AI_Data_Quality_RootCause_Copilot/blob/main/AI_Data_Quality_RootCause_Copilot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Data Quality & Root-Cause Analysis Copilot

**Business requirement:** analyze datasets and pipeline issues, identify data-quality problems, investigate root causes, and recommend fixes -- in plain business language, with SQL/Python corrective suggestions.



## Cell 1 -- Setup

In [1]:
# Install what we need (Gradio for the interactive UI, google-genai for Gemini)
!pip install -q pandas numpy matplotlib gradio google-genai

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)  # so results are reproducible every run

## Cell 2 -- Load real data, simulate a broken pipeline run, generate a matching ETL log

In [2]:
# ---- Run 1: baseline, straight from Kaggle's Titanic dataset ----
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
baseline_df = pd.read_csv(url)

# ---- Run 2: what a BROKEN ETL pipeline actually hands you ----
df = baseline_df.copy()
log_lines = []
t0 = datetime.now().replace(microsecond=0)

def log(t_off, level, stage, msg):
    # helper to write one ETL log line, timestamped t_off seconds after pipeline start
    ts = (t0 + timedelta(seconds=t_off)).strftime('%Y-%m-%d %H:%M:%S')
    log_lines.append(f"{ts} {level} etl.{stage} - {msg}")

# 1) Duplicates -- a retry-after-timeout bug re-inserted rows
dupes = df.sample(15, random_state=1)
df = pd.concat([df, dupes], ignore_index=True)
log(3, "WARN", "load", f"retry after timeout on batch load; {len(dupes)} rows re-inserted (possible duplicate keys)")

# 2) Outliers -- a unit-conversion bug multiplied Fare by 100 for a few rows
outlier_idx = df.sample(5, random_state=2).index
df.loc[outlier_idx, 'Fare'] = df.loc[outlier_idx, 'Fare'] * 100
log(7, "ERROR", "validate", f"{len(outlier_idx)} rows failed range check on column 'Fare'")

# 3) Invalid records -- a null-handling bug cast missing ages to -1 instead of NaN
invalid_idx = df.sample(4, random_state=3).index
df.loc[invalid_idx, 'Age'] = -1
log(10, "ERROR", "validate", f"{len(invalid_idx)} rows failed constraint check on column 'Age' (negative not allowed)")

# 4) Schema change -- source system renamed a column + changed a dtype upstream
df = df.rename(columns={'Embarked': 'Port_Embarked'})
df['PassengerId'] = df['PassengerId'].astype(str)
log(15, "ERROR", "schema", "column 'Embarked' missing; unexpected column 'Port_Embarked' found")
log(15, "ERROR", "schema", "dtype mismatch on 'PassengerId': expected int64, got object")

corrupted_df = df
etl_log_text = "\n".join(log_lines)

print(etl_log_text)
print("\nBaseline shape:", baseline_df.shape, "| Corrupted shape:", corrupted_df.shape)

2026-09-06 12:43:06 WARN etl.load - retry after timeout on batch load; 15 rows re-inserted (possible duplicate keys)
2026-09-06 12:43:10 ERROR etl.validate - 5 rows failed range check on column 'Fare'
2026-09-06 12:43:13 ERROR etl.validate - 4 rows failed constraint check on column 'Age' (negative not allowed)
2026-09-06 12:43:18 ERROR etl.schema - column 'Embarked' missing; unexpected column 'Port_Embarked' found
2026-09-06 12:43:18 ERROR etl.schema - dtype mismatch on 'PassengerId': expected int64, got object

Baseline shape: (891, 12) | Corrupted shape: (906, 12)


## Cell 3 -- Automated data profiling module

In [3]:
def profile_dataframe(df):
    """Returns a dict summary: shape, dtypes, nulls, duplicates, IQR-based outliers."""
    profile = {}
    profile['shape'] = df.shape
    profile['dtypes'] = df.dtypes.astype(str).to_dict()

    null_counts = df.isnull().sum()
    profile['nulls'] = {c: int(n) for c, n in null_counts.items() if n > 0}
    profile['null_pct'] = {c: round(n / len(df) * 100, 2) for c, n in null_counts.items() if n > 0}

    profile['duplicate_rows'] = int(df.duplicated().sum())

    # IQR method: flag values far outside the normal spread, per numeric column
    outliers = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        cnt = int(((df[col] < lo) | (df[col] > hi)).sum())
        if cnt > 0:
            outliers[col] = cnt
    profile['outliers_iqr'] = outliers
    return profile

baseline_profile = profile_dataframe(baseline_df)
corrupted_profile = profile_dataframe(corrupted_df)

print("=== Corrupted run profile ===")
for k, v in corrupted_profile.items():
    print(f"{k}: {v}")

=== Corrupted run profile ===
shape: (906, 12)
dtypes: {'PassengerId': 'object', 'Survived': 'int64', 'Pclass': 'int64', 'Name': 'object', 'Sex': 'object', 'Age': 'float64', 'SibSp': 'int64', 'Parch': 'int64', 'Ticket': 'object', 'Fare': 'float64', 'Cabin': 'object', 'Port_Embarked': 'object'}
nulls: {'Age': 181, 'Cabin': 698, 'Port_Embarked': 2}
null_pct: {'Age': 19.98, 'Cabin': 77.04, 'Port_Embarked': 0.22}
duplicate_rows: 14
outliers_iqr: {'Age': 8, 'SibSp': 46, 'Parch': 215, 'Fare': 122}


## Cell 4 -- Data Quality Rule Engine

In [4]:
# Define business rules once; the engine checks every row against them
RULES = [
    {"column": "Age",      "rule": "min_value", "params": {"min": 0},   "description": "Age must not be negative"},
    {"column": "Fare",     "rule": "max_value", "params": {"max": 600}, "description": "Fare must be within a realistic historical range"},
    {"column": "Sex",      "rule": "in_set",    "params": {"allowed": {"male", "female"}}, "description": "Sex must be male/female"},
    {"column": "Pclass",   "rule": "in_set",    "params": {"allowed": {1, 2, 3}}, "description": "Pclass must be 1, 2 or 3"},
    {"column": "PassengerId", "rule": "unique", "params": {}, "description": "PassengerId must be unique"},
]

def evaluate_rules(df, rules):
    results = []
    for r in rules:
        col, rtype, params, desc = r["column"], r["rule"], r["params"], r["description"]
        if col not in df.columns:
            results.append({"column": col, "description": desc, "status": "SKIPPED", "violations": None})
            continue
        series = df[col]
        if rtype == "min_value":
            bad = series < params["min"]
        elif rtype == "max_value":
            bad = series > params["max"]
        elif rtype == "in_set":
            bad = ~series.isin(params["allowed"])
        elif rtype == "unique":
            bad = series.duplicated(keep=False)
        else:
            bad = pd.Series([False] * len(df))
        violations = int(bad.sum())
        results.append({
            "column": col, "description": desc,
            "status": "PASS" if violations == 0 else "FAIL",
            "violations": violations,
            "sample_rows": df[bad].index[:5].tolist() if violations else []
        })
    return pd.DataFrame(results)

rule_report = evaluate_rules(corrupted_df, RULES)
print(rule_report.to_string(index=False))

     column                                      description status  violations              sample_rows
        Age                         Age must not be negative   FAIL           4     [487, 740, 784, 898]
       Fare Fare must be within a realistic historical range   FAIL           5 [37, 183, 447, 455, 733]
        Sex                          Sex must be male/female   PASS           0                       []
     Pclass                         Pclass must be 1, 2 or 3   PASS           0                       []
PassengerId                       PassengerId must be unique   FAIL          30        [2, 3, 6, 17, 34]


## Cell 5 -- Schema & Format Change Detector

In [5]:
def capture_schema(df):
    return {col: str(dtype) for col, dtype in df.dtypes.items()}

def diff_schema(baseline_schema, current_schema):
    added = set(current_schema) - set(baseline_schema)
    removed = set(baseline_schema) - set(current_schema)
    common = set(baseline_schema) & set(current_schema)
    dtype_changes = {c: (baseline_schema[c], current_schema[c]) for c in common if baseline_schema[c] != current_schema[c]}
    return {"columns_added": sorted(added), "columns_removed": sorted(removed), "dtype_changes": dtype_changes}

baseline_schema = capture_schema(baseline_df)
current_schema = capture_schema(corrupted_df)
schema_diff = diff_schema(baseline_schema, current_schema)
print(schema_diff)

{'columns_added': ['Port_Embarked'], 'columns_removed': ['Embarked'], 'dtype_changes': {'PassengerId': ('int64', 'object')}}


## Cell 6 -- ETL Pipeline Log Analyzer

In [6]:
import re
LOG_PATTERN = re.compile(r"^(?P<ts>\S+ \S+) (?P<level>\w+) (?P<stage>\S+) - (?P<msg>.+)$")

def parse_etl_log(log_text):
    records = [m.groupdict() for line in log_text.strip().split("\n") if (m := LOG_PATTERN.match(line))]
    return pd.DataFrame(records)

def summarize_log(log_df):
    return {
        "total_entries": len(log_df),
        "by_level": log_df["level"].value_counts().to_dict(),
        "by_stage": log_df["stage"].value_counts().to_dict(),
        "errors": log_df[log_df["level"] == "ERROR"]["msg"].tolist(),
    }

log_df = parse_etl_log(etl_log_text)
log_summary = summarize_log(log_df)
print(log_summary)

{'total_entries': 5, 'by_level': {'ERROR': 4, 'WARN': 1}, 'by_stage': {'etl.validate': 2, 'etl.schema': 2, 'etl.load': 1}, 'errors': ["5 rows failed range check on column 'Fare'", "4 rows failed constraint check on column 'Age' (negative not allowed)", "column 'Embarked' missing; unexpected column 'Port_Embarked' found", "dtype mismatch on 'PassengerId': expected int64, got object"]}


## Cell 7 -- GenAI layer (Gemini): root-cause reasoning, business explanation, fix suggestions\n\nGet a free API key from https://aistudio.google.com/app/apikey before running this cell.

In [7]:
import time
from google import genai
from google.genai import errors as genai_errors

GEMINI_API_KEY = input("Paste your Gemini API key: ").strip()
client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-3.6-flash"  # current model; 2.5-flash was retired for new API keys

def ask_gemini(prompt, max_retries=4):
    """Send a prompt to Gemini and return plain text. Retries with backoff on transient 503s."""
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            return response.text
        except genai_errors.ServerError:
            wait = 2 ** attempt  # 1, 2, 4, 8 seconds
            print(f"Gemini overloaded (attempt {attempt + 1}/{max_retries}), retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Gemini API still unavailable after retries -- wait a few minutes and try again.")

def build_findings_summary():
    lines = [
        f"Dataset shape: {corrupted_profile['shape']}",
        f"Null columns: {corrupted_profile['null_pct']}",
        f"Duplicate rows: {corrupted_profile['duplicate_rows']}",
        f"Outliers (IQR): {corrupted_profile['outliers_iqr']}",
        "", "Rule engine results:", rule_report.to_string(index=False),
        "", f"Schema changes: {schema_diff}",
        "", "ETL log errors:",
    ]
    lines += [f"- {e}" for e in log_summary['errors']]
    return "\n".join(lines)

def analyze_root_cause():
    findings = build_findings_summary()
    prompt = f"""You are a Data Quality & Root-Cause Analysis Copilot helping a non-technical business stakeholder.

Here are the automated findings from today's pipeline run:
{findings}

Respond with 4 clearly labeled sections:
1. BUSINESS EXPLANATION - what went wrong, in plain non-technical language.
2. LIKELY ROOT CAUSE - for each issue, your best guess at the underlying pipeline cause.
3. FIX SUGGESTIONS - one SQL snippet and one Python/pandas snippet to correct these issues.
4. PREVENTION - 2-3 concrete recommendations to stop this recurring."""
    return ask_gemini(prompt)

root_cause_report = analyze_root_cause()
print(root_cause_report)

Paste your Gemini API key: AQ.Ab8RN6JHObeAtqMNI5UiPCP_jzeEVQAwE3iynIUSCiOjot7arw
Gemini overloaded (attempt 1/4), retrying in 1s...
Gemini overloaded (attempt 2/4), retrying in 2s...
Gemini overloaded (attempt 3/4), retrying in 4s...
Here is the data quality assessment and root-cause analysis based on today's pipeline run.

---

### 1. BUSINESS EXPLANATION
**What went wrong today:**
Today’s automated data intake suffered from several critical data flaws that make this dataset unsafe for immediate business reporting or analytics:

* **Missing Key Location Info:** The system could not find the standard `Embarked` column (location where passengers boarded). It appears under a new name (`Port_Embarked`), which will break existing downstream dashboards expecting the original column name.
* **Double-Counting Risk & Customer ID Corruption:** Passenger ID numbers changed format from standard numbers to text/symbols and contain **30 non-unique IDs** alongside **14 completely duplicate rows**. T

## Cell 8 -- Interactive Gradio interface

In [8]:
import gradio as gr

def generate_full_report():
    return f"""# Data Quality & Root-Cause Analysis Report

## Automated Findings
{build_findings_summary()}

## AI Analysis
{analyze_root_cause()}
"""

def chat_fn(message, history):
    prompt = f"""You are a Data Quality Copilot. Use these findings to answer the user's question.
Findings:
{build_findings_summary()}

Question: {message}
Answer in plain business language."""
    return ask_gemini(prompt)

with gr.Blocks(title="Data Quality & Root-Cause Copilot") as demo:
    gr.Markdown("# AI Data Quality & Root-Cause Analysis Copilot")
    with gr.Tab("Full Report"):
        run_btn = gr.Button("Run Full Analysis")
        report_box = gr.Markdown()
        run_btn.click(fn=generate_full_report, outputs=report_box)
    with gr.Tab("Ask the Copilot"):
        gr.ChatInterface(fn=chat_fn)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://279a56ac2ef79524bc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
